# 문항 2. 타입 가드 분기 처리와 제네릭 저장소

## in 연산자 기반 타입 좁히기

`User | Product` 유니온 타입을 `in` 연산자로 좁혀서 안전하게 분기 처리합니다. 이 노트북은 단독으로 실행되도록 문항 1의 타입을 다시 정의합니다.

In [1]:
// 셀 타입 다시 정의
type UserRole = 'admin' | 'customer' | 'guest';
type Category = 'hamster-doll' | 'cat-doll' | 'otter-doll';

interface BaseEntity {
  id: number;
}

interface User extends BaseEntity {
  name: string;
  email: string;
  role: UserRole;
  isActive: boolean;
}

interface Product extends BaseEntity {
  title: string;
  price: number;
  category: Category;
  stock: number;
  discountRate?: number;
}

function formatWon(amount: number): string {
  return amount.toLocaleString('ko-KR') + '원';
}

In [2]:
// 샘플 데이터 

const users: User[] = [
  { id: 1, name: '김관리', email: 'admin@test.com', role: 'admin', isActive: true },
  { id: 2, name: '이고객', email: 'lee@test.com', role: 'customer', isActive: true },
];

const products: Product[] = [
  { id: 101, title: '하치와레', price: 80000, category: 'cat-doll', stock: 15, discountRate: 0.1 },
  { id: 102, title: '쿠리만쥬', price: 90000, category: 'otter-doll', stock: 40 },
];

In [3]:
// renderItemSummary (in 연산자로 좁히기)

function renderItemSummary(item: User | Product): string {
  if ('email' in item) {
    // 이 블록 안에서 item은 User로 좁혀진다.
    return `[회원] ${item.name} (${item.role})`;
  } else {
    // 이 블록 안에서 item은 Product로 좁혀진다.
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
}

In [4]:
// 실행과 결과 확인

users.forEach((u) => console.log(renderItemSummary(u)));
products.forEach((p) => console.log(renderItemSummary(p)));

const mixed: (User | Product)[] = [users[0], products[0], users[1], products[1]];
console.log('--- 섞인 배열 ---');
mixed.forEach((item) => console.log(renderItemSummary(item)));

[회원] 김관리 (admin)
[회원] 이고객 (customer)
[상품] 하치와레 80,000원
[상품] 쿠리만쥬 90,000원
--- 섞인 배열 ---
[회원] 김관리 (admin)
[상품] 하치와레 80,000원
[회원] 이고객 (customer)
[상품] 쿠리만쥬 90,000원


### 가드 없이 접근하면 막히는지 확인

`@ts-expect-error`는 "다음 줄은 오류가 나야 정상"이라는 표시입니다. 셀이 에러 없이 끝나면 컴파일러가 두 가지 위험한 접근을 모두 막았다는 의미입니다.

In [5]:
// 의도적 오류

function wrongAccess(item: User | Product): string {
  // @ts-expect-error 가드 없이 email에 접근하면 Product에는 없으므로 오류가 나야 정상
  return item.email;
}

function wrongBranch(item: User | Product): string {
  if ('email' in item) {
    // @ts-expect-error User 블록 안에서 Product 전용 title에 접근하면 오류가 나야 정상
    return item.title;
  }
  return item.title;
}

console.log('가드 밖/잘못된 분기 접근 2개를 컴파일러가 막았습니다.');

가드 밖/잘못된 분기 접근 2개를 컴파일러가 막았습니다.


### 모든 케이스를 처리했는지 확인 (exhaustive)

유니온의 모든 경우를 처리하고 나면 남는 타입은 `never`입니다. `never`만 받는 `assertNever`를 마지막에 두면 새 타입이 추가됐을 때 처리를 빼먹으면 컴파일 오류가 납니다.

In [6]:
// exhaustive 버전

function assertNever(value: never): never {
  throw new Error(`처리되지 않은 케이스: ${JSON.stringify(value)}`);
}

function renderItemSummaryStrict(item: User | Product): string {
  if ('email' in item) {
    return `[회원] ${item.name} (${item.role})`;
  }
  if ('title' in item) {
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
  return assertNever(item);
}

console.log(renderItemSummaryStrict(users[0]));
console.log(renderItemSummaryStrict(products[1]));

[회원] 김관리 (admin)
[상품] 쿠리만쥬 90,000원


In [7]:
// 새 타입을 빼먹으면 컴파일러가 알려준다.

interface Coupon extends BaseEntity {
  code: string;
  amountOff: number;
}

function renderWithCoupon(item: User | Product | Coupon): string {
  if ('email' in item) {
    return `[회원] ${item.name} (${item.role})`;
  }
  if ('title' in item) {
    return `[상품] ${item.title} ${formatWon(item.price)}`;
  }
  // @ts-expect-error Coupon 케이스를 처리하지 않아 item이 never가 아니므로 오류가 나야 정상
  return assertNever(item);
}

console.log('새 타입(Coupon)을 처리하지 않으면 컴파일러가 알려줍니다.');

새 타입(Coupon)을 처리하지 않으면 컴파일러가 알려줍니다.


## 심화: 제네릭 제약 조건을 적용한 InMemoryStorage

모든 요소가 고유한 `id`를 갖도록 `Identifiable` 제약을 걸고
어떤 타입이든(제약을 만족하면) 저장할 수 있는 범용 저장소 클래스를 만듭니다.

In [8]:
// 제약 인터페이스

interface Identifiable {
    id: number;
}

In [9]:
// 제네릭 클래스

class InMemoryStorage<T extends Identifiable> {
  private items: T[] = [];

  add(item: T): void {
    this.items.push(item);
  }

  getById(id: number): T | undefined {
    return this.items.find((item) => item.id === id);
  }

  getAll(): T[] {
    return [...this.items];
  }
}

In [10]:
// 확인

const quickCheck = new InMemoryStorage<User>();
quickCheck.add(users[0]);
quickCheck.add(users[1]);
console.log('조회(id=1):', quickCheck.getById(1)?.name);
console.log('조회(id=999):', quickCheck.getById(999));
console.log('전체 개수:', quickCheck.getAll().length);

조회(id=1): 김관리
조회(id=999): undefined
전체 개수: 2


### 다형성 인스턴스 검증

같은 `InMemoryStorage` 클래스로 회원 저장소와 상품 저장소를 각각 만들고 제네릭 타입 추론이 올바르게 동작하는지 확인합니다.

In [11]:
// 인스턴스 생성, 추가, 조회

const userStorage = new InMemoryStorage<User>();
const productStorage = new InMemoryStorage<Product>();

users.forEach((user) => userStorage.add(user));
products.forEach((product) => productStorage.add(product));

console.log('--- 회원 저장소 ---');
console.log('전체 회원 수:', userStorage.getAll().length);
const foundUser = userStorage.getById(2);
if (foundUser) {
  console.log('id=2 조회:', renderItemSummary(foundUser), '/ 이메일:', foundUser.email);
}
console.log('id=999 조회:', userStorage.getById(999));

console.log('--- 상품 저장소 ---');
console.log('전체 상품 수:', productStorage.getAll().length);
const foundProduct = productStorage.getById(101);
if (foundProduct) {
  console.log('id=101 조회:', renderItemSummary(foundProduct), '/ 재고:', foundProduct.stock);
}
console.log('id=999 조회:', productStorage.getById(999));

--- 회원 저장소 ---
전체 회원 수: 2
id=2 조회: [회원] 이고객 (customer) / 이메일: lee@test.com
id=999 조회: undefined
--- 상품 저장소 ---
전체 상품 수: 2
id=101 조회: [상품] 하치와레 80,000원 / 재고: 15
id=999 조회: undefined


### 제약 위반과 잘못된 사용이 막히는지 확인

아래 함수는 **일부러 호출하지 않습니다.** 컴파일러가 코드를 검사하는 것만 확인하기 위해서입니다. `@ts-expect-error`가 붙은 줄도 실행하면 실제로 동작하기 때문에, 저장소가 오염되지 않도록 함수 안에 두었습니다.

In [12]:
// 의도적 오류

function showTypeErrors(): void {
  // @ts-expect-error User 저장소에 Product를 넣으면 오류가 나야 정상
  userStorage.add(products[0]);

  // @ts-expect-error Product 저장소에서 꺼낸 값에는 email이 없어 오류가 나야 정상
  productStorage.getById(101)?.email;

  // @ts-expect-error id가 없는 string은 Identifiable 제약을 만족하지 못해 오류가 나야 정상
  const badStorage = new InMemoryStorage<string>();

  // @ts-expect-error private items에는 클래스 바깥에서 접근할 수 없어 오류가 나야 정상
  userStorage.items;
}

console.log('제약 위반과 잘못된 사용 4개를 컴파일러가 막았습니다.');

제약 위반과 잘못된 사용 4개를 컴파일러가 막았습니다.


In [13]:
// 복사 반환 확인과 다른 타입에 재사용

const snapshot = userStorage.getAll();
snapshot.pop();
console.log('복사본 길이:', snapshot.length);
console.log('저장소 길이:', userStorage.getAll().length);

const couponStorage = new InMemoryStorage<Coupon>();
couponStorage.add({ id: 1, code: 'WELCOME', amountOff: 5000 });
console.log('쿠폰 조회:', couponStorage.getById(1)?.code);

복사본 길이: 1
저장소 길이: 2
쿠폰 조회: WELCOME


### 제네릭 저장소 설계 근거

**1.제약 조건으로 안전성 확보**

- `T extends Identifiable`로 모든 요소가 `id: number`를 갖도록 보장했고 `id`가 없는 `string`으로는 저장소를 만들 수 없음을 컴파일러가 막는 것을 확인했습니다.

**2. 하나의 클래스로 재사용**

- 회원(`User`), 상품(`Product`), 쿠폰(`Coupon`)이 클래스 수정 없이 같은 저장소 로직을 씁니다.

**3. 캡슐화**

- 내부 배열 `items`를 `private`으로 두어 바깥에서 직접 접근하면 컴파일 오류가 납니다.

**4. 복사 반환**

- `getAll()`이 `[...this.items]` 복사본을 돌려줘서, 받은 배열을 고쳐도 저장소 내용은 그대로입니다.

**5. 타입 추론**

- `getById`가 `T | undefined`를 돌려주므로 못 찾은 경우를 타입으로 드러내고, 찾은 값은 `User`/`Product` 그대로 사용할 수 있습니다.